In [ ]:
from _utils_siamese import *

In [ ]:
def __changeshape(imgs, new_shape):
    if len(imgs.shape) == 3:
        resize_factor = [a / b for a, b in zip(new_shape, imgs.shape)]
        imgnew = zoom(imgs, resize_factor, mode='nearest')
        return imgnew
    else:
        raise ValueError('Wrong shape, expected 3D array')


def load_and_preprocess_image(image_path, target_size=None):
    img = nib.load(image_path).get_fdata()
    
    # Adjust axis order: assume original is (H, W, D) -> change to (D, H, W)
    img = np.transpose(img, (2, 0, 1))  # Added transpose operation
    original_shape = img.shape
    
    # Keep the original high-resolution image's axis order
    original_img = normalize(img)  # Shape (D, H, W)

    if target_size:
        img_resized = __changeshape(img, target_size)
    else:
        img_resized = img

    img_normalized = normalize(img_resized)
    img_normalized = img_normalized[np.newaxis, np.newaxis, ...]  # (1,1,D,H,W)
    
    return img_normalized, original_img, original_shape

In [ ]:
def make_gradcam_heatmap(model, input_data, target_layer_name, input_index=0, pred_index=None):
    if isinstance(input_data, np.ndarray):
        input_data = tf.convert_to_tensor(input_data, dtype=tf.float32)

    target_layer = model.get_layer(target_layer_name)
    grad_model = Model(inputs=model.input, outputs=[target_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(input_data)
        conv_outputs, predictions = grad_model(input_data)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]

    grads = tape.gradient(loss, conv_outputs)[0]  # Gradient shape (channels, D, H, W)
    
    # take the mean across spatial dimensions for each channel
    weights = tf.reduce_mean(grads, axis=(1, 2, 3))  

    # Weighted feature map
    conv_outputs = conv_outputs[0]  # Shape (channels, D, H, W)
    heatmap = np.zeros(conv_outputs.shape[1:], dtype=np.float32)  
    
    # iterate over the channel dimension
    for i, w in enumerate(weights):
        heatmap += w * conv_outputs[i, :, :, :]  

    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap) + 1e-8
    return heatmap

def resize_heatmap(heatmap, original_shape):
    # Fix scaling factor correspondence
    scale_factors = [
        original_shape[0]/heatmap.shape[0],  # D axis
        original_shape[1]/heatmap.shape[1],  # H axis
        original_shape[2]/heatmap.shape[2]   # W axis
    ]
    return zoom(heatmap, scale_factors, order=1)

def display_heatmap(original_img, heatmap, original_shape, title=None, slice_idx=None, save_path=None):
    heatmap_resized = resize_heatmap(heatmap, original_shape)
    
    if slice_idx is None:
        slice_idx = original_shape[0] // 2
    
    raw_slice = original_img[slice_idx, :, :]
    heatmap_slice = heatmap_resized[slice_idx, :, :]

    # Rotate 90 degrees clockwise (k=3) and horizontally flip
    rotated_img = np.rot90(raw_slice, k=3)
    rotated_img = np.fliplr(rotated_img)  # Horizontal flip
    rotated_img = np.flipud(rotated_img)  # Vertical flip 
    
    rotated_heatmap = np.rot90(heatmap_slice, k=3)
    rotated_heatmap = np.fliplr(rotated_heatmap)
    rotated_heatmap = np.flipud(rotated_heatmap)  
    
    plt.figure(figsize=(12,6))
    plt.subplot(121)
    plt.imshow(rotated_img, cmap='gray')
    plt.axis('off')  
    # plt.title("Original")
    
    plt.subplot(122)
    plt.imshow(rotated_img, cmap='gray')
    plt.imshow(rotated_heatmap, cmap='jet', alpha=0.5)
    plt.axis('off')  
    # plt.title("Grad-CAM")
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()


In [ ]:
# load model
Size = (64, 64, 64) 
weight_decay = 1e-4  
dropout_rate = 0.5   
Class_num = 2      

model_path = ".../final_epoch_fold2_model.h5"   

model = build_CNN(Size, weight_decay, dropout_rate, Class_num)
model.load_weights(model_path)

In [ ]:

image_path = ".../116_trans_pre_cropped.nii.gz"   

target_size = (64, 64, 64)

processed_img, original_img, original_shape = load_and_preprocess_image(image_path, target_size)

heatmap = make_gradcam_heatmap(
    model=model,
    input_data=processed_img,
    target_layer_name="conv3d_2"  
)

display_heatmap(
    original_img=original_img,  
    heatmap=heatmap,
    original_shape=original_shape,
    slice_idx= 6,
    save_path= ".../gradcam.png"
)